# 2 · From positions to speed

**What this notebook does:** Gestures are movements, so what matters most is *how fast* the hands move. This
notebook turns the positions from Notebook 1 into a speed for each body point you choose. On the way it fills
in small gaps, puts all measurements on an even clock, and smooths away tracking jitter.

| You need | You get |
|---|---|
| `Keypoints/Test_Video_keypoints.csv` (from Notebook 1) | `Upsampled/Test_Video_processed_data.csv`: position, speed and direction of movement of each chosen body point, every 10 milliseconds. |

*Reminder:* click a code cell and press **Shift + Enter** to run it. You only need to change the **Settings** cell.

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import numpy as np                       # maths on long lists of numbers
import pandas as pd                      # tables
import plotly.graph_objects as go        # interactive plots
from scipy.signal import savgol_filter   # the smoothing filter

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"

# Which body points to calculate speed for. Use the names from Notebook 1, for example
# "right_wrist", "left_wrist", "right_index" (right index finger), "right_elbow", "nose"
KEYPOINTS = ["right_wrist", "right_index"]

# Make one measurement every STEP_MS milliseconds (10 ms = 100 measurements per second)
STEP_MS = 10

# How strongly to smooth: how many measurements the filter looks at together.
# Must be an odd number. 11 measurements x 10 ms = a window of 110 ms.
SMOOTHING_WINDOW = 11

# Speeds slower than this percentile count as tracking jitter and are set to 0 ("not moving").
# 50 means: the slower half of all moments in the video counts as rest.
JITTER_PERCENTILE = 50

## Step 3 · Load the positions from Notebook 1

In [ ]:
keypoints = pd.read_csv(f"../Keypoints/{VIDEO_NAME}_keypoints.csv")

print(f"Loaded {len(keypoints)} video frames")
position_columns = [f"{point}_{axis}" for point in KEYPOINTS for axis in ["x", "y"]]
keypoints[["time_ms"] + position_columns].head()

## Step 4 · Put the data on an even 10 ms clock

Video frames come about every 33 ms (30 frames per second), and some phone videos have slightly uneven timing.
We'd like one measurement every 10 ms instead, for every video. To get there we draw a straight line between
the positions in neighbouring frames and read off the value every 10 ms. This is called **interpolation**
(and because we end up with more measurements than before, **upsampling**).

The same trick fills in the frames where no person was found.

In [ ]:
# The new, even clock: 0, 10, 20, 30, ... milliseconds, up to the end of the video
data = pd.DataFrame({"time_ms": np.arange(0, keypoints["time_ms"].iloc[-1], STEP_MS)})

for column in position_columns:
    tracked = keypoints[column].notna()  # the frames where this point was found
    data[column] = np.interp(data["time_ms"],                    # read off a value at these times,
                             keypoints.loc[tracked, "time_ms"],  # using the tracked frames' times
                             keypoints.loc[tracked, column])     # and positions

print(f"{len(keypoints)} frames became {len(data)} measurements, one every {STEP_MS} ms")

## Step 5 · Smooth away jitter

Even when a hand is still, its tracked position wobbles a little from frame to frame. A **Savitzky-Golay filter**
smooths this out: for every measurement it takes its neighbours (`SMOOTHING_WINDOW` of them) and fits a gentle
curve through them. Real movements survive, tiny wobbles don't.

In [ ]:
raw = data.copy()  # keep the unsmoothed version, to compare in the plot below
for column in position_columns:
    data[column] = savgol_filter(data[column], SMOOTHING_WINDOW, polyorder=3)

column = position_columns[1]  # the y position of the first body point
fig = go.Figure()
fig.add_scatter(x=raw["time_ms"], y=raw[column], name="before smoothing", line_color="lightgrey")
fig.add_scatter(x=data["time_ms"], y=data[column], name="after smoothing", line_color="royalblue")
fig.update_layout(title=f"{column}: before and after smoothing (zoom in to see the difference)",
                  xaxis_title="Time (ms)", yaxis_title="Position (0 = top of the video)")
fig.update_xaxes(range=[0, 10_000])     # start by showing the first 10 seconds
fig.update_yaxes(autorange="reversed")  # higher in the plot = higher in the video
fig.show()

## Step 6 · Calculate speed

For each body point we calculate:

* **Velocity** (columns ending in `_delta_x` and `_delta_y`): how much x and y change per millisecond.
  The sign gives the direction: positive x = moving to the right of the picture, positive y = moving down.
* **Speed** (column ending in `_speed`): how fast the point moves in any direction. It combines x and y with
  Pythagoras' rule: speed = √(x velocity² + y velocity²).

Then the speed gets three clean-up steps:

1. **Scale to 0-1:** divide by the fastest moment in the video, so 1 = the fastest the point ever moves.
   This makes speakers who make big movements and speakers who make small ones comparable.
2. **Remove jitter:** speeds below the `JITTER_PERCENTILE` are set to 0, meaning "not moving".
3. **Smooth once more**, so the speed curve is clean.

In [ ]:
for point in KEYPOINTS:
    velocity_x = np.gradient(data[f"{point}_x"], STEP_MS)  # change in x per millisecond
    velocity_y = np.gradient(data[f"{point}_y"], STEP_MS)
    speed = np.sqrt(velocity_x**2 + velocity_y**2)

    speed = speed / speed.max()                                  # 1. scale: 1 = fastest moment
    speed[speed < np.percentile(speed, JITTER_PERCENTILE)] = 0   # 2. tiny movements count as rest
    speed = savgol_filter(speed, SMOOTHING_WINDOW, polyorder=3)  # 3. smooth again...
    speed = np.clip(speed, 0, None)                              # ...but speed can't go below 0

    data[f"{point}_speed"] = speed
    data[f"{point}_delta_x"] = velocity_x
    data[f"{point}_delta_y"] = velocity_y

# Put the columns in a tidy order: time, then everything about each body point
columns = ["time_ms"] + [f"{point}_{measure}" for point in KEYPOINTS
                         for measure in ["x", "y", "speed", "delta_x", "delta_y"]]
data = data[columns]
data.head()

## Step 7 · Save

In [ ]:
Path("../Upsampled").mkdir(exist_ok=True)
output_path = f"../Upsampled/{VIDEO_NAME}_processed_data.csv"
data.to_csv(output_path, index=False)
print("Saved:", output_path)

## Step 8 · Look at the speed

Each hill is a burst of movement, and most of them are gestures. Use the slider under the plot to move through
the video, and click a name in the legend to hide or show that line.

In [ ]:
fig = go.Figure()
for point in KEYPOINTS:
    fig.add_scatter(x=data["time_ms"], y=data[f"{point}_speed"], name=point)
fig.update_layout(title="Speed over time", xaxis_title="Time (ms)",
                  yaxis_title="Speed (1 = fastest)", xaxis_rangeslider_visible=True)
fig.show()

## Done!

Next, either:
* **Notebook 3a** finds the gestures automatically, or
* **Notebook 3b** finds the apex of strokes that you annotated by hand in ELAN.